# 03 · 벌크 vs 계면 분리 — 회절패턴 NMF (공간 분해)

계면(interface)은 **공간적 특징**이라 스캔 전체를 평균하면 벌크에 묻힙니다. 계면을 보려면 **위치별 회절
패턴을 그대로** 분해해야 합니다.

**회절패턴은 음수가 없으므로 NMF가 정확**하고 parts-based라 물리적 상 분리에 적합합니다(이게 4D-STEM
NMF의 원래 용도). 각 온도의 4D를 `X (n_position × n_detpix)`로 펼쳐 `X ≈ W · H` 로 분해:
- **H(성분)** = 상별 회절패턴 (벌크 / 계면)
- **W(loading)** = **실공간 지도** → 계면이 어디 띠로 있는지 보임
- 각 성분패턴 → `pattern_to_rdf` → **벌크 RDF · 계면 RDF**
- 온도별 **계면 성분의 양** → 계면이 사라지는 곡선

> 메모리: 4D 하나를 위치별로 NMF하면 큽니다(150×150×256×256). `DET_BIN`으로 **검출기를 비닝**해서
> `n_detpix`를 줄이세요(예: 4 → 64×64). 스캔은 그대로라 계면 지도 해상도는 유지됩니다.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DATA_ROOT = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiO"
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)

N_COMPONENTS   = 2         # 벌크 + 계면 (필요시 3으로 — 벌크/계면/기타)
DET_BIN        = 4         # 검출기 비닝(메모리). 실데이터 256→64. 합성이면 1
Q_UNIT_HINT    = "1/A"
BEAM_RADIUS_PX = 12        # (비닝 후 기준으로 자동 축소됨)
Q_PER_PX       = 0.0120    # 01의 2b 캘리브레이션 값 넣기 (비닝하면 ×DET_BIN 되어 자동 반영)
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")
print("USE_SYNTHETIC =", USE_SYNTHETIC)

## 1) 한 온도의 4D 큐브 불러오기 (+ 검출기 비닝)

실데이터: `fds.load(파일)` → `(sy,sx,qy,qx)`. 합성: 벌크(가운데 링) + 세로 계면 띠(다른 링)로 만든 데모.

In [ ]:
def make_interface_cube(scan=(60, 60), dp=(64, 64), iface_cols=(26, 34),
                        bulk_r=16.0, iface_r=22.0, seed=0):
    '''벌크(링 bulk_r) + 세로 계면 띠(iface_cols에 링 iface_r 추가) 4D 데모.'''
    rng = np.random.default_rng(seed)
    Sy, Sx = scan; H, W = dp
    yy, xx = np.mgrid[0:H, 0:W]; cx, cy = W/2, H/2
    rr = np.hypot(xx - cx, yy - cy)
    bulk  = np.exp(-(rr-bulk_r)**2/(2*3.5**2)) + 3*np.exp(-rr**2/(2*2**2))
    iface = np.exp(-(rr-iface_r)**2/(2*3.0**2)) + 0.6*np.exp(-(rr-bulk_r)**2/(2*3.5**2)) + 3*np.exp(-rr**2/(2*2**2))
    cube = np.empty((Sy, Sx, H, W), np.float32)
    for ix in range(Sx):
        pat = iface if (iface_cols[0] <= ix < iface_cols[1]) else bulk
        cube[:, ix] = pat + 0.02*rng.standard_normal((Sy, H, W))
    return np.clip(cube, 0, None)

if USE_SYNTHETIC:
    cube = fds.from_array(make_interface_cube(), q_per_px=Q_PER_PX, name="synthetic")
    DET_BIN = 1
else:
    f0 = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))[0]   # 예: 최저온
    cube = fds.load(f0, Q_UNIT_HINT)
    print("loaded", f0, cube.shape)

if DET_BIN > 1:
    cube = fds.bin_cube_detector(cube, DET_BIN)     # 검출기 축소(메모리)
qpp = (cube.calibration.q_per_px or Q_PER_PX)
print("cube for NMF:", cube.shape, "| q_per_px:", qpp)

## 1b) 먼저 가상 이미지로 스캔 확인 (계면이 보이나?)

⚠️ **주의**: 위치별 총세기는 빔커런트/두께/스캔라인 드리프트로 변합니다 → 가상이미지(ADF 등)에 **가로
줄무늬(스캔 아티팩트)** 로 나타납니다. 계면(구조 차이)을 보려면 **밝기로 정규화**해야 합니다:
`구조링 DF / 총세기`. 정규화 지도에서 세로 띠가 보이면 그게 계면입니다.

In [ ]:
mean_dp = cube.mean_dp()
(cx, cy), _ = fds.find_center(mean_dp, fds.beam_stopper_mask(mean_dp))
dp = cube.dp_shape
adf   = fds.annular_dark_field(cube, center=(cx, cy),
                               r_inner=min(dp)/6, r_outer=min(dp)/2)
ringdf = cube.get_virtual_image(fds.annular_mask(dp, (cx, cy), min(dp)/5, min(dp)/3))
total  = cube.get_virtual_image(fds.annular_mask(dp, (cx, cy), min(dp)/12, min(dp)/2))
struct = ringdf / np.where(total > 0, total, 1.0)     # 밝기 정규화된 구조 대비

fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
ax[0].imshow(adf, cmap="viridis"); ax[0].axis("off")
ax[0].set_title("ADF (raw): horizontal stripes = scan artifact")
im = ax[1].imshow(struct, cmap="viridis"); ax[1].axis("off")
ax[1].set_title("brightness-normalized structural map (interface?)")
plt.colorbar(im, ax=ax[1], fraction=0.046); plt.tight_layout(); plt.show()

## 2) 회절패턴 NMF → 성분 패턴 + 실공간 지도

**패턴별 정규화(`normalize="sum"`)** 로 밝기(스캔 아티팩트)를 제거하고 **구조로** 분해합니다. 중심빔+beam
stop을 마스크해 정규화가 산란(구조) 세기를 반영하게 합니다. 결과:
- `components (k, qy, qx)` = 상별 회절패턴
- `loadings (k, sy, sx)` = 실공간 abundance 지도 (**계면 = 세로 띠**여야 함)

In [ ]:
stopper = fds.beam_stopper_mask(mean_dp)
beam = fds.disk_mask(dp, (cx, cy), max(2, BEAM_RADIUS_PX // max(DET_BIN, 1)))
mask = fds.combine_masks(stopper, beam)                  # 중심빔 + stopper 제외
res = fds.nmf_decompose(cube, n_components=N_COMPONENTS, mask=mask,
                        normalize="sum")                 # ★ 밝기 아닌 구조로 분해
print("components:", res.components.shape, " loadings(maps):", res.loadings.shape)

# 계면 성분 식별: 실공간 지도가 더 '좁게 뭉친' 성분 (localization = max/mean 비 높은 쪽)
loc = [float(np.nanmax(m) / (np.nanmean(m) + 1e-9)) for m in res.loadings]
iface_idx = int(np.argmax(loc)); bulk_idx = 1 - iface_idx if N_COMPONENTS == 2 else None
print("localization:", np.round(loc, 2), "→ 계면 성분 =", iface_idx)

fig, axes = plt.subplots(2, N_COMPONENTS, figsize=(4.2*N_COMPONENTS, 8))
for j in range(N_COMPONENTS):
    tag = "INTERFACE" if j == iface_idx else ("bulk" if j == bulk_idx else f"comp{j}")
    axes[0][j].imshow(np.log1p(res.components[j]), cmap="magma")
    axes[0][j].set_title(f"{tag}: diffraction"); axes[0][j].axis("off")
    axes[1][j].imshow(res.loadings[j], cmap="viridis")
    axes[1][j].set_title(f"{tag}: real-space map"); axes[1][j].axis("off")
plt.tight_layout(); plt.show()

## 3) 각 성분의 RDF — 벌크 RDF vs 계면 RDF

NMF 성분 **패턴(비음수)** 을 각각 `pattern_to_rdf`에 넣어 상별 G(r)를 얻습니다. 계면상과 벌크상의
**구조 차이(결합거리·중거리 질서)** 를 직접 비교할 수 있습니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for j in range(N_COMPONENTS):
    comp = res.components[j]
    (cx, cy), _ = fds.find_center(comp, fds.beam_stopper_mask(comp))
    rr = fds.pattern_to_rdf(comp, qpp, CFG, center=(cx, cy),
                            center_beam_radius=max(1, BEAM_RADIUS_PX // DET_BIN))
    tag = "INTERFACE" if j == iface_idx else ("bulk" if j == bulk_idx else f"comp{j}")
    ax[0].plot(rr.q, rr.Iq/np.nanmax(rr.Iq), label=tag)
    ax[1].plot(rr.r, rr.Gr, label=tag)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("I(q) norm"); ax[0].set_title("component I(q)"); ax[0].legend()
ax[1].axhline(0, color="0.8", lw=0.8); ax[1].set_xlim(0, 6)
ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)"); ax[1].set_title("component RDF: bulk vs interface"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4) 계면이 온도에 따라 사라지는가 — 온도별 계면 비율

각 온도의 4D를 NMF해서 **계면 성분의 총량 비율**(= 계면 loading 합 / 전체)을 온도의 함수로 봅니다.
비율이 온도↑ 에 따라 **감소→0** 이면 계면이 사라지는 것. (실데이터는 큐브가 커서 온도당 수 초~수십 초.)

In [ ]:
def interface_fraction_of(cube_bin, k=N_COMPONENTS):
    md_ = cube_bin.mean_dp(); stop = fds.beam_stopper_mask(md_)
    (cxx, cyy), _ = fds.find_center(md_, stop)
    m = fds.combine_masks(stop, fds.disk_mask(cube_bin.dp_shape, (cxx, cyy),
                          max(2, BEAM_RADIUS_PX // max(DET_BIN, 1))))
    rr = fds.nmf_decompose(cube_bin, n_components=k, mask=m, normalize="sum")
    loc = [float(np.nanmax(mm)/(np.nanmean(mm)+1e-9)) for mm in rr.loadings]
    ii = int(np.argmax(loc))
    tot = np.array([mm.sum() for mm in rr.loadings])
    return tot[ii] / (tot.sum() + 1e-9)

if USE_SYNTHETIC:
    # 데모: 온도↑ → 계면 띠 폭 축소(28→30 ... 점점 좁게) → 사라짐
    Ts = [300, 500, 700, 900, 1100]
    fracs = []
    for T in Ts:
        w = max(0, int(round(8 * (1 - (T-300)/900))))     # 폭 8→0
        c0 = 30 - w//2
        cu = make_interface_cube(iface_cols=(c0, c0+w) if w > 0 else (0, 0), seed=T)
        fracs.append(interface_fraction_of(fds.from_array(cu, q_per_px=Q_PER_PX)))
else:
    files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.dm4")))
    Ts, fracs = [], []
    for p in files:
        T = fds.coordinate_from_name(os.path.splitext(os.path.basename(p))[0])
        cb = fds.load(p, Q_UNIT_HINT)
        if DET_BIN > 1: cb = fds.bin_cube_detector(cb, DET_BIN)
        Ts.append(T); fracs.append(interface_fraction_of(cb))
        print(f"  T={T:>6.0f}K  interface fraction = {fracs[-1]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(Ts, fracs, "o-", color="crimson")
ax.set_xlabel("temperature (K)"); ax.set_ylabel("interface component fraction")
ax.set_title("interface disappearing vs temperature")
plt.show()

**정리** — 벌크/계면은 **평균 G(r)이 아니라 위치별 회절패턴 NMF**로 분리합니다. 성분 패턴 → RDF로 상별
구조를, loading 지도로 계면 위치를, 온도별 계면 비율로 소멸 거동을 봅니다.
- `N_COMPONENTS=3`: 벌크/계면 외 제3상(예: 결정핵·기공)이 있으면 늘려보세요.
- `DET_BIN`: 실데이터 메모리에 맞게(4~8). 계면 지도(스캔)는 영향 없음.
- 성분↔상 매칭은 loading **지도를 눈으로** 확인해 `iface_idx`가 맞는지 검증하세요.